In [ ]:
import torch
from diffusers import StableDiffusion3Pipeline, EulerDiscreteScheduler, EulerAncestralDiscreteScheduler
from PIL import Image
from config import MODEL_CACHE, SEED, TSR_SIGMA, SWAP_ALGORITHM, N_INF_STEPS, GUIDANCE_SCALE

tsr_lam = 1.0
replica_exchange = False

pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    torch_dtype=torch.float16,
    cache_dir=MODEL_CACHE,
).to("cuda")

prompt = "a photo of a cat sitting on a chair"
seed = 42
n_samples = 4

def generate_samples(use_sde, seed, n=4):
    if use_sde:
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    else:
        pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)
    
    images = []
    for i in range(n):
        generator = torch.Generator("cuda").manual_seed(seed + i)
        img = pipe(
            prompt,
            negative_prompt="",
            num_inference_steps=N_INF_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            tsr_lam=tsr_lam,
            tsr_sigma=TSR_SIGMA,
            replica_exchange=replica_exchange,
            swap_algorithm=SWAP_ALGORITHM,
            generator=generator,
        ).images[0]
        images.append(img)
    return images

ode_images = generate_samples(use_sde=False, seed=seed)
sde_images = generate_samples(use_sde=True,  seed=seed)

In [ ]:
import matplotlib.pyplot as plt

def show_comparison(ode_images, sde_images):
    fig, axes = plt.subplots(2, len(ode_images), figsize=(4*len(ode_images), 8))
    
    for i, img in enumerate(ode_images):
        axes[0, i].imshow(img)
        axes[0, i].axis("off")
        axes[0, i].set_title(f"ODE #{i+1}" if i > 0 else "ODE (deterministic)")

    for i, img in enumerate(sde_images):
        axes[1, i].imshow(img)
        axes[1, i].axis("off")
        axes[1, i].set_title(f"SDE #{i+1}" if i > 0 else "SDE (stochastic)")

    plt.suptitle("Same seed — ODE identical, SDE varies", fontsize=14)
    plt.tight_layout()
    plt.show()

show_comparison(ode_images, sde_images)